# Reliable Text-to-SQL Modeling on MIMIC III dataset with schema

This Jupyter notebook serves as a comprehensive guide to run Text-to-SQL model for Electronic Health Records (EHRs).

## Steps in This Jupyter Notebook
- [x] Step 1: Initial Setup
- [x] Step 2: Load and Prepare Datasets
- [x] Step 3: Construct a Text-to-SQL Model
- [x] Step 4: Model Evaluation

## Getting Started

Begin your journey with the EHRSQL task by following these structured steps (from Step 1 to Step 8). Each section is designed to guide you smoothly through the process, from setup to submission. We're eager to see the innovative solutions you'll bring to the field of Text-to-SQL modeling for electronic health records.

## Step 1: Initial Setup

As part of the intial setup we will pull code to Colab environment, install required Python dependencies and mount a GCP bucket to store intermediate results.

Clone the GitHub Repository created for this project.  It is a modified version of the original EHRSQL repo created by the writers of the paper.

In [2]:
%cd /content
!rm -rf EHRSQL

%ls -al

# Cloning the GitHub repository
!git clone -q https://github.com/BizUnix/EHRSQL.git
%cd EHRSQL



/content
total 16
drwxr-xr-x 1 root root 4096 Apr  3 13:37 ./
drwxr-xr-x 1 root root 4096 Apr  6 12:21 ../
drwxr-xr-x 4 root root 4096 Apr  3 13:37 .config/
drwxr-xr-x 1 root root 4096 Apr  3 13:37 sample_data/
/content/EHRSQL


Install Required Python Packages:

In [3]:
# Installing dependencies

!pip install -q transformers
!pip install -q sentencepiece
!pip install -q func_timeout


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Use the `%load_ext` magic command to automatically reload modules before executing a new line:

In [3]:
%load_ext autoreload
%autoreload 2

Mount a GCP bucket for storing intermediate results and output

In [4]:
# prompt: get gcsfuse

!apt-get update -qq
!apt-get install -qq -y fuse
!curl -LO https://github.com/googlecloudplatform/gcsfuse/releases/download/v1.0.1/gcsfuse_1.0.1_amd64.deb
!dpkg -i gcsfuse_1.0.1_amd64.deb




W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 5072k  100 5072k    0     0  3124k      0  0:00:01  0:00:01 --:--:-- 76.2M
Selecting previously unselected package gcsfuse.
(Reading database ... 126213 files and directories currently installed.)
Preparing to unpack gcsfuse_1.0.1_amd64.deb ...
Unpacking gcsfuse (1.0.1) ...
Setting up gcsfuse (1.0.1) ...


In [5]:
from google.colab import auth;
auth.authenticate_user()

# Replace with your project ID and bucket name
!gcloud config set project striking-yen-455401-m9
!gcsfuse -v

!mkdir -p /content/cse6250_h1

# Mount bucket
!gcsfuse cse6250_h1 /content/cse6250_h1




Updated property [core/project].
gcsfuse version 1.0.1 (Go version go1.20.5)
I0406 12:40:41.468100 2025/04/06 12:40:41.468063 Start gcsfuse/1.0.1 (Go version go1.20.5) for app "" using mount point: /content/cse6250_h1


## Step 2: Load Data and Prepare Datasets

Now that we have our environment and paths set up, the next step is to load the data and prepare it for our model.  This involves preprocessing the MIMIC-III database and storing it into SQLLITE, reading the data from JSON files, splitting it into training and validation sets, and then initializing our dataset object.

### Preprocess MIMIC-III Data

Use a secure copy of the MIMIC-III data on Google Cloud Platform bucket to preprocess data and create the SQLLITE Mimic III dataset.


In [ ]:
%cd preprocess
!python preprocess_db.py --data_dir "/content/cse6250_h1/mimic-iii" --db_name "mimic_iii"
%cd ..

/content/ehrsql/preprocess
Processing PATIENTS, ADMISSIONS, ICUSTAYS, TRANSFERS
num_cur_patient: 0
num_non_cur_patient: 1000
num_patient: 1000
PATIENTS, ADMISSIONS, ICUSTAYS, TRANSFERS processed (took 15.0231 secs)
Processing D_ICD_DIAGNOSES, D_ICD_PROCEDURES, D_LABITEMS, D_ITEMS
D_ICD_DIAGNOSES, D_ICD_PROCEDURES, D_LABITEMS, D_ITEMS processed (took 2.0864 secs)
Processing DIAGNOSES_ICD table
DIAGNOSES_ICD processed (took 2.9386 secs)
Processing PROCEDURES_ICD table
PROCEDURES_ICD processed (took 2.2436 secs)
Processing LABEVENTS table
LABEVENTS processed (took 200.0603 secs)
Processing PRESCRIPTIONS table
PRESCRIPTIONS processed (took 55.0977 secs)
Processing COST table
COST processed (took 5.326 secs)
Processing CHARTEVENTS table
[########################################] | 100% Completed | 24m 45s
CHARTEVENTS processed (took 1496.1085 secs)
Processing INPUTEVENTS_CV table
INPUTEVENTS_CV processed (took 165.6402 secs)
Processing OUTPUTEVENTS table
OUTPUTEVENTS processed (took 35.9414

### Save SQLLite DB to GCP Bucket

This will help skip the preprocess step upon restarts


In [ ]:

# Move SQLLite to Google Bucket
!cp -r dataset/ehrsql/mimic_iii/mimic_iii.sql* /content/cse6250_h1/


## Step 3: Construct a Text-to-SQL Baseline Model

In this step, we set up and train a T5 model to translate natural language queries into SQL statements. The process involves several key stages including argument parsing, model initialization, data preparation, and the actual training.


### Check GPU and connect to Google Drive (to store output)

In [6]:
import torch
print(f"Torch Version: {torch.__version__}")

if torch.cuda.is_available():
    gpu_id = torch.cuda.current_device()
    print(f"GPU ID: {gpu_id}")
    print(f"GPU Name: {torch.cuda.get_device_name(gpu_id)}")
else:
    print("No GPU detected!")

Torch Version: 2.6.0+cu124
GPU ID: 0
GPU Name: NVIDIA A100-SXM4-40GB


In [7]:
# Check if the files /content/ehrsql/dataset/ehrsql/mimic_iii/mimic_iii.sql* exist
# if they don't download them from Google Cloud Platform bucket
!ls dataset/ehrsql/mimic_iii*

!if [ ! -f "dataset/ehrsql/mimic_iii/mimic_iii.sqlite" ]; then cp -r /content/cse6250_h1/mimic_iii.sqlite dataset/ehrsql/mimic_iii/; fi

!ls dataset/ehrsql/mimic_iii*

mimic_iii.sql  test.json  train.json  valid.json
mimic_iii.sql  mimic_iii.sqlite  test.json  train.json	valid.json


### Training the (T5) Model

Finally, we train the model on the dataset. The training process involves learning to generate SQL queries from textual descriptions through iterative forward and backward passes, loss computation, and parameter updates.  This version of the training use the pretrained T5 base model and trains it on the EHRSQL dataset without schema.

In [8]:
!rm -rf outputs/ehrsql_mimic3_t5_base_schema
!ls outputs

eval_ehrsql_mimic3_t5_base__mimic3_valid  eval_ehrsql_mimic3_t5_base_schema__mimic3_valid


In [9]:
!python T5/main.py --config T5/config/ehrsql/training/ehrsql_mimic3_t5_base_schema.yaml --CUDA_VISIBLE_DEVICES 0 --model_name t5-base


Current device: cuda:0
2025-04-06 12:41:11 | INFO | Namespace(exp_name='ehrsql_mimic3_t5_base_schema', load_model_path=None, device='cuda', num_workers=50, random_seed=0, report_every_step=50, eval_batch_size=8, save_every_step=-1, save_every_epoch=False, show_eval_sample=True, eval_every_step=5000, eval_metric='loss', keep_last_ckpt=-1, early_stop_patience=-1, training_data_ratio=1.0, bf16=False, use_wandb=False, wandb_project=None, dataset='ehrsql', db_id='mimic_iii', train_data_path='dataset/ehrsql/mimic_iii/train.json', valid_data_path='dataset/ehrsql/mimic_iii/valid.json', output_dir='outputs', output_file='prediction_raw.json', model_name='t5-base', db_path=None, add_schema=True, add_column_type=False, shuffle_schema=False, tables_path='dataset/ehrsql/tables.json', condition_value=True, warmup_steps=0, total_epoch=-1, total_step=100000, train_batch_size=4, accumulation_steps=8, lr=0.0001, scheduler_steps=None, optim='adam', scheduler='fixed', max_grad_norm=1.0, weight_decay=0.1, 

### Store Model Output to GCP Bucket

In [10]:
#copy model output to Google Cloud Bucket
!cp -r outputs/ehrsql_mimic3_t5_base_schema /content/cse6250_h1/outputs


In [11]:
!ls -al outputs/ehrsql_mimic3_t5_base_schema

total 2612732
drwxr-xr-x 2 root root       4096 Apr  6 12:58 .
drwxr-xr-x 5 root root       4096 Apr  6 12:41 ..
-rw-r--r-- 1 root root 2675193510 Apr  6 15:18 checkpoint_best.pth.tar
-rw-r--r-- 1 root root     222803 Apr  6 18:29 train.log


## Step 4: Model Evaluation

In this step, we will evaluate the model's performance across all queries, using the Reliability Score (RS) as our evaluation metric. This will provide a baseline understanding of the model's reliability scroe without filtering for unanswerable queries.

### Prepare for Running Inference

Load model from Google Drive to output directory, load SQL Database and get ready to run inference

In [ ]:
#if directory outputs/ehrsql_mimic3_t5_base doesn't exist, create it
!mkdir -p outputs/ehrsql_mimic3_t5_base_schema

#copy model from GCP bucket to output directory
!cp -r /content/cse6250_h1/outputs/ehrsql_mimic3_t5_base_schema/* outputs/ehrsql_mimic3_t5_base_schema

#copy SQL lite database from Google cloud
!cp -r /content/cse6250_h1/mimic_iii.sql* dataset/ehrsql/mimic_iii/


In [12]:
# List output
!ls outputs/ehrsql_mimic3_t5_base_schema/*

# cleanup output directory for inferences
!rm -rf outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid
!rm -rf outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test

!ls -al outputs/ehrsql_mimic3_t5_base_schema


outputs/ehrsql_mimic3_t5_base_schema/checkpoint_best.pth.tar
outputs/ehrsql_mimic3_t5_base_schema/train.log
total 2612732
drwxr-xr-x 2 root root       4096 Apr  6 12:58 .
drwxr-xr-x 4 root root       4096 Apr  6 18:41 ..
-rw-r--r-- 1 root root 2675193510 Apr  6 15:18 checkpoint_best.pth.tar
-rw-r--r-- 1 root root     222803 Apr  6 18:29 train.log


### Run inference with Validation Data

In [13]:
# Run inferences for validation data
!python T5/main.py --config T5/config/ehrsql/eval/ehrsql_mimic3_t5_base_schema__mimic3_valid.yaml --output_file prediction_raw.json --CUDA_VISIBLE_DEVICES 0


Current device: cuda:0
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
100% 1122/1122 [00:00<00:00, 14756.17it/s]
loaded 1122 test examples from dataset/ehrsql/mimic_iii/valid.json
2025-04-06 18:42:01.303985: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-06 18:42:01.320887: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register c

### Run Inference with Test Data

In [17]:
# Run inferences for test data
!python T5/main.py --config T5/config/ehrsql/eval/ehrsql_mimic3_t5_base_schema__mimic3_test.yaml --output_file prediction_raw.json --CUDA_VISIBLE_DEVICES 0


Current device: cuda:0
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
100% 1786/1786 [00:00<00:00, 14962.24it/s]
loaded 1786 test examples from dataset/ehrsql/mimic_iii/test.json
2025-04-06 19:04:53.277936: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-06 19:04:53.294328: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cu

### Run abstention with or without threshold to generate final files

We save these predictions to a JSON file in a designated results directory, creating the directory if necessary.  The JSON files are then moved to GCP to allow for generating performance metrics.

In [18]:
# Run abstention wihtout threshold for validation data
!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid --input_file prediction_raw.json --output_file prediction.no_threshold.json --threshold -1

# Run abstention with threshold of 0.14923561
!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid --input_file prediction_raw.json --output_file prediction.fixed_threshold.json --threshold 0.14923561

# Backup all predictions to GCP buucket
!cp -r outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid /content/cse6250_h1/outputs


/content/EHRSQL/T5/abstain_with_entropy.py:18: UserWarning: Threshold value is not set! All predictions are sent to the database.
  warnings.warn("Threshold value is not set! All predictions are sent to the database.")


In [19]:
# Run abstention wihtout threshold for test data
!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test --input_file prediction_raw.json --output_file prediction.no_threshold.json --threshold -1

# Run abstention with threshold of 0.14923561
!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test --input_file prediction_raw.json --output_file prediction.fixed_threshold.json --threshold 0.14923561

# Backup all predictions to GCP buucket
!cp -r outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test /content/cse6250_h1/outputs


/content/EHRSQL/T5/abstain_with_entropy.py:18: UserWarning: Threshold value is not set! All predictions are sent to the database.
  warnings.warn("Threshold value is not set! All predictions are sent to the database.")


### Evaluate Performance


#### With Validation Data

In this step we evaluate performance of the results produced in prior steps with the JSON produced for Validation Data

In [20]:
# Check if the output already exists on
# outputs/eval_ehrsql_mimic3_t5_base__mimic3_test if not copy it from GCP- this
# step allows for repeatable runs without running inference

!if [ ! -f "outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid" ]; then cp -r /content/cse6250_h1/outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid/ outputs/; fi

!echo "Key Metrics for Validation Data - with schema"

!echo "With No Threshold"
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/valid.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid/prediction.no_threshold.json

!echo "With Fixed Threshold"
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/valid.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid/prediction.fixed_threshold.json



Key Metrics for Validation Data - with schema
With No Threshold
{
  "precision_ans": 67.74,
  "recall_ans": 100.0,
  "f1_ans": 80.77,
  "precision_exec": 65.51,
  "recall_exec": 96.71,
  "f1_exec": 78.11
}
With Fixed Threshold
{
  "precision_ans": 94.27,
  "recall_ans": 80.13,
  "f1_ans": 86.63,
  "precision_exec": 92.72,
  "recall_exec": 78.82,
  "f1_exec": 85.21
}


#### With Test Data

In this step we evaluate performance of the results produced in prior steps with the JSON produced for Test Data

In [21]:
# Check if the output already exists on
# outputs/eval_ehrsql_mimic3_t5_base__mimic3_test if not copy it from GCP- this
# step allows for repeatable runs without running inference

!if [ ! -f "outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test" ]; then cp -r /content/cse6250_h1/outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test/ outputs/; fi

!echo "Key Metrics for Test Data - with schema"

!echo "With No Threshold"
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/test.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test/prediction.no_threshold.json

!echo "With Fixed Threshold"
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/test.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test/prediction.fixed_threshold.json



Key Metrics for Test Data - with schema
With No Threshold
{
  "precision_ans": 67.08,
  "recall_ans": 100.0,
  "f1_ans": 80.29,
  "precision_exec": 64.73,
  "recall_exec": 96.49,
  "f1_exec": 77.48
}
With Fixed Threshold
{
  "precision_ans": 88.3,
  "recall_ans": 81.3,
  "f1_ans": 84.66,
  "precision_exec": 87.31,
  "recall_exec": 80.38,
  "f1_exec": 83.7
}
